# Movie Review Sentiment Analysis — Neural Network Models

This notebook trains three deep learning models (Simple NN, CNN, LSTM) on the IMDb 50K reviews dataset using **GloVe word embeddings** to classify reviews as positive or negative.

**Files needed:**
- `IMDB_Dataset.csv` — [Kaggle link](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)
- `glove.6B.100d.txt` — [GloVe link](https://nlp.stanford.edu/projects/glove/)
- `IMDb_Unseen_Reviews.csv` — sample reviews for live prediction

## 1. Imports

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Flatten, Dropout, Activation,
    Embedding, Conv1D, GlobalMaxPooling1D, LSTM
)


## 2. Load Dataset

In [ ]:
df = pd.read_csv('IMDB_Dataset.csv')

print('Shape:', df.shape)
df.head()


## 3. Sentiment Distribution

In [ ]:
sns.countplot(x='sentiment', data=df)
plt.title('Positive vs Negative Reviews')
plt.show()


## 4. Text Preprocessing

We strip HTML tags, remove non-alpha characters, single characters, extra spaces, and English stopwords.

In [ ]:
nltk.download('stopwords')

TAG_RE = re.compile(r'<[^>]+>')

def clean_text(text):
    text = text.lower()
    text = TAG_RE.sub('', text)                          # remove HTML tags
    text = re.sub('[^a-zA-Z]', ' ', text)               # keep only letters
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)        # remove lone chars
    text = re.sub(r'\s+', ' ', text)                    # collapse spaces
    stop_pattern = re.compile(
        r'\b(' + r'|'.join(stopwords.words('english')) + r')\b\s*'
    )
    text = stop_pattern.sub('', text)
    return text


## 5. Apply Cleaning & Encode Labels

In [ ]:
X = [clean_text(review) for review in df['review']]

# encode: positive → 1, negative → 0
y = np.array([1 if label == 'positive' else 0 for label in df['sentiment']])

print('Sample cleaned review:')
print(X[2])


## 6. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')


## 7. Tokenisation & Padding

Keras `Tokenizer` maps each word to an integer ID. We then pad all sequences to a fixed length of 100 tokens.

In [ ]:
MAX_LEN = 100

tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, padding='post', maxlen=MAX_LEN)
X_test_pad  = pad_sequences(X_test_seq,  padding='post', maxlen=MAX_LEN)

vocab_size = len(tokenizer.word_index) + 1
print('Vocabulary size:', vocab_size)


## 8. Load GloVe Embeddings (100-d)

GloVe pre-trained vectors give the model a head start by initialising word representations with semantic meaning.

In [ ]:
embeddings = {}

with open('glove.6B.100d.txt', encoding='utf8') as f:
    for line in f:
        parts  = line.split()
        word   = parts[0]
        vector = np.asarray(parts[1:], dtype='float32')
        embeddings[word] = vector

print(f'Loaded {len(embeddings):,} word vectors')


## 9. Build Embedding Matrix

In [ ]:
EMBED_DIM = 100

embed_matrix = np.zeros((vocab_size, EMBED_DIM))
for word, idx in tokenizer.word_index.items():
    vec = embeddings.get(word)
    if vec is not None:
        embed_matrix[idx] = vec

print('Embedding matrix shape:', embed_matrix.shape)


---
## Model Training

All three models share the same frozen GloVe embedding layer. We track `accuracy` (not `acc`) so history keys are consistent across TensorFlow versions.

### Plot helper

In [ ]:
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(history.history['accuracy'],     label='train')
    ax1.plot(history.history['val_accuracy'], label='val')
    ax1.set_title(f'{title} — Accuracy')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend()

    ax2.plot(history.history['loss'],     label='train')
    ax2.plot(history.history['val_loss'], label='val')
    ax2.set_title(f'{title} — Loss')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend()

    plt.tight_layout()
    plt.show()


## 10. Model 1 — Simple Neural Network

In [ ]:
def build_embedding_layer():
    return Embedding(
        vocab_size, EMBED_DIM,
        weights=[embed_matrix],
        input_length=MAX_LEN,
        trainable=False
    )

snn = Sequential([
    build_embedding_layer(),
    Flatten(),
    Dense(1, activation='sigmoid')
])

snn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
snn.summary()


In [ ]:
snn_hist = snn.fit(
    X_train_pad, y_train,
    batch_size=128, epochs=6,
    validation_split=0.2, verbose=1
)

loss, acc = snn.evaluate(X_test_pad, y_test, verbose=0)
print(f'\nSNN  →  Loss: {loss:.4f} | Accuracy: {acc:.4f}')
plot_history(snn_hist, 'Simple NN')


## 11. Model 2 — Convolutional Neural Network

In [ ]:
cnn = Sequential([
    build_embedding_layer(),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(1, activation='sigmoid')
])

cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.summary()


In [ ]:
cnn_hist = cnn.fit(
    X_train_pad, y_train,
    batch_size=128, epochs=6,
    validation_split=0.2, verbose=1
)

loss, acc = cnn.evaluate(X_test_pad, y_test, verbose=0)
print(f'\nCNN  →  Loss: {loss:.4f} | Accuracy: {acc:.4f}')
plot_history(cnn_hist, 'CNN')


## 12. Model 3 — LSTM

In [ ]:
lstm = Sequential([
    build_embedding_layer(),
    LSTM(128),
    Dense(1, activation='sigmoid')
])

lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm.summary()


In [ ]:
lstm_hist = lstm.fit(
    X_train_pad, y_train,
    batch_size=128, epochs=6,
    validation_split=0.2, verbose=1
)

loss, acc = lstm.evaluate(X_test_pad, y_test, verbose=0)
print(f'\nLSTM  →  Loss: {loss:.4f} | Accuracy: {acc:.4f}')
plot_history(lstm_hist, 'LSTM')

lstm.save(f'lstm_model_acc_{round(acc, 3)}.h5')


## 13. Live Predictions on Unseen Reviews

In [ ]:
sample_df = pd.read_csv('IMDb_Unseen_Reviews.csv')
sample_df.head(6)


In [ ]:
cleaned = [clean_text(r) for r in sample_df['Review Text']]
tokenized = tokenizer.texts_to_sequences(cleaned)
padded    = pad_sequences(tokenized, padding='post', maxlen=MAX_LEN)

predictions = lstm.predict(padded)

sample_df['Predicted Sentiment Score'] = np.round(predictions * 10, 1)
print(sample_df[['Movie', 'IMDb Rating', 'Predicted Sentiment Score']])

sample_df.to_csv('IMDb_Predictions_Output.csv', index=False)
